In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import datetime

# Cấu hình giao diện đồ thị
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
path="/home/slow_data/Air_Quality/filtered_envisoft_air_quality_weather_data.csv"

In [ ]:
df_aqi = pd.read_csv(path)
df_aqi.head()

In [ ]:
df_aqi.info()

In [ ]:
QB_filter = df_aqi[df_aqi['Name'] == 'Quảng Bình: KKT Hòn La (KK)']
QB_filter.head()

In [ ]:
filtered_df_loc = df_aqi.loc[df_aqi['ID'] == '29213751141295132066317063859']

In [ ]:
filtered_df_loc.head()

# Extract AOD for stations

In [ ]:
import rasterio
import glob
from  pathlib import Path

In [ ]:
unique_stations = df_aqi['Name'].unique()
station_df = df_aqi.groupby('Name')[['ID','Latitude', 'Longitude']].first().reset_index()


In [ ]:
station_df

In [ ]:
station_df.to_csv('/home/slow_data/Air_Quality/Envisoft_station_metadata.csv', index=False)

In [ ]:
aod_dir = "/home/slow_data/Air_Quality/AOD/L2_aod"
OUTPUT_DIR = "/home/slow_data/Air_Quality/AOD/station_aod"

aod_path = Path(aod_dir)

In [ ]:
uncertainty_thresholds = [0.5]

In [ ]:
output_files = [OUTPUT_DIR + f"/all_station{str(threshold).replace('.', '')}.csv" for threshold in uncertainty_thresholds]


In [ ]:
pattern = os.path.join(aod_dir, r'2025??/*/*/aod_vietnam_NC_H??_*_L2ARP031_FLDK.*.tif')
print(pattern)
files = glob.glob(pattern)
print(files)

for aod_file in files:
    print("found file: ", aod_file)
    # Lấy timestamp từ tên file
    filename = os.path.basename(aod_file)
    parts = filename.split("_")
    timestamp = parts[4] + "_" + parts[5]

    # Đọc ảnh AOT và uncertainty
    with rasterio.open(aod_file) as src:
        # Đọc toàn bộ band 1 và band 2 trước để tăng tốc
        aot_band = src.read(1)
        uncertainty_band = src.read(2)
        
        # Tạo dict để lưu giá trị cho từng ngưỡng
        aot_values_dict = {threshold: [] for threshold in uncertainty_thresholds}
        
        for _, row in station_df.iterrows():
            lon, lat = row["Longitude"], row["Latitude"]
            try:
                rowcol = src.index(lon, lat)
                aot_value = aot_band[rowcol[0], rowcol[1]]
                uncertainty_value = uncertainty_band[rowcol[0], rowcol[1]]
                
                for threshold in uncertainty_thresholds:
                    if np.isnan(aot_value) or np.isnan(uncertainty_value) or uncertainty_value >= threshold:
                        aot_values_dict[threshold].append(np.nan) 
                    else:
                        aot_values_dict[threshold].append(float(aot_value))
            except Exception as e:
                for threshold in uncertainty_thresholds:
                    aot_values_dict[threshold].append(np.nan)

    # Lưu ra các file CSV tương ứng với từng ngưỡng uncertainty
    for threshold, output_csv in zip(uncertainty_thresholds, output_files):
        # Tạo DataFrame cho hàng mới: timestamp + AOT values cho từng station
        new_row = {'timestamp': timestamp}
        
        # Thêm giá trị AOT cho từng station (dùng station_id gốc làm tên cột)
        for i, station_id in enumerate(station_df['ID']):
            new_row[str(station_id)] = aot_values_dict[threshold][i]
        
        new_df = pd.DataFrame([new_row])
        
        # Thêm vào file CSV (append)
        if os.path.exists(output_csv):
            df_old = pd.read_csv(output_csv)
            # Đảm bảo tất cả các cột (trừ timestamp) là kiểu float
            for col in df_old.columns:
                if col != 'timestamp':
                    df_old[col] = df_old[col].astype('float64')
                    new_df[col] = new_df[col].astype('float64')
            
            df_merged = pd.concat([df_old, new_df], ignore_index=True)
        else:
            df_merged = new_df
        
        df_merged.to_csv(output_csv, index=False)

    print(f"✅ Hoàn tất xử lý cho timestamp {timestamp}")

In [ ]:
# df_merged.head()

--------

In [ ]:
df_aod = pd.read_csv("/home/slow_data/Air_Quality/AOD/station_aod/all_station05.csv")

In [ ]:
df_aod.head()

In [ ]:
# # Deduplicate timestamps: keep the row with the most non-null station values for each timestamp
# cols = df_AOD.columns.drop('timestamp')
# df_AOD['non_null_count'] = df_AOD[cols].notna().sum(axis=1)

# df_AOD_dedup = (
#     df_AOD.sort_values(['timestamp', 'non_null_count'], ascending=[True, False])
#           .drop_duplicates('timestamp', keep='first')
#           .drop(columns='non_null_count')
#           .reset_index(drop=True)
# )

# # overwrite in-memory variable and save a deduplicated CSV alongside the original
# df_AOD = df_AOD_dedup

# df_AOD.to_csv("/home/slow_data/Air_Quality/AOD/station_aod/all_station05.csv", index=False)

# df_AOD.head()

--------

# Process and Merge

In [ ]:
# A. Process AOD Data (10 min -> Hourly Mean)
# Convert timestamp: '20250101_0000' -> Datetime
df_aod['timestamp'] = pd.to_datetime(df_aod['timestamp'], format='%Y%m%d_%H%M')

# Set index to resample
df_aod.set_index('timestamp', inplace=True)

# Resample to Hourly (H) and take the Mean
# This calculates the mean of the 6 data points per hour per station, ignoring NaNs
df_aod_hourly = df_aod.resample('h').mean()

# Reset index to make timestamp a column again
df_aod_hourly.reset_index(inplace=True)

# Melt AOD from Wide (Columns are IDs) to Long (Rows are IDs)
# 'var_name' becomes the name of the column holding the Station IDs
df_aod_long = df_aod_hourly.melt(id_vars=['timestamp'], var_name='ID', value_name='AOD')


In [ ]:
df_aod_long

In [ ]:
# B. Process AQI Data
# Convert timestamp: '08/04/2025 14:00' -> Datetime
df_aqi['Timestamp_DT'] = pd.to_datetime(df_aqi['Timestamp'], format='%d/%m/%Y %H:%M')

df_aqi

In [ ]:
# C. CRITICAL STEP: Ensure IDs are Strings
# Sometimes headers load as integers while the other file has strings.
# We force both to strings to ensure they match during merge.
df_aqi['ID'] = df_aqi['ID'].astype(str)
df_aod_long['ID'] = df_aod_long['ID'].astype(str)

# D. Merge Datasets
df_merged = pd.merge(
    df_aqi, 
    df_aod_long, 
    left_on=['Timestamp_DT', 'ID'], 
    right_on=['timestamp', 'ID'], 
    how='inner'
)



print(f"Data merged successfully. Total paired records: {len(df_merged)}")
print("Sample of merged data:")
print(df_merged[['Timestamp_DT', 'ID', 'AQI', 'AOD']].head())

-------------------------

# Graph

In [ ]:
import matplotlib.dates as mdates

In [ ]:

# Process: Select one station to visualize
target_station = '31390916083317566102523755051'

# df_aod is the wide table (columns are station IDs), df_aod_long is the long table (rows per ID).
# Handle both cases robustly.
if target_station in df_aod.columns:
	# wide format: column exists
	df_plot = df_aod[[target_station]].copy()
	df_plot.columns = ['AOD']
	# ensure datetime index
	if not isinstance(df_plot.index, pd.DatetimeIndex):
		df_plot.index = pd.to_datetime(df_plot.index)
else:
	# long format: filter by ID
	df_plot = df_aod_long[df_aod_long['ID'] == target_station][['timestamp', 'AOD']].copy()
	if not df_plot.empty:
		df_plot['timestamp'] = pd.to_datetime(df_plot['timestamp'])
		df_plot.set_index('timestamp', inplace=True)
	else:
		# station not found in either — create empty time index from df_aod to avoid KeyError and allow plotting
		df_plot = pd.DataFrame(index=df_aod.index)
		df_plot.index = pd.to_datetime(df_plot.index)
		df_plot['AOD'] = np.nan

# Important: To plot gaps, we must keep NaNs.
# Resample to hourly (this will produce NaNs for hours with no data)
df_hourly = df_plot.resample('H').mean()

In [ ]:
df_plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
plt.subplots_adjust(hspace=0.4)

# --- Plot 1: Full Time Series (12 Months / Full Range) ---
# Matplotlib automatically breaks the line at NaNs
ax1.plot(df_hourly.index, df_hourly['AOD'], color='tab:blue', linewidth=1.5, label=f'Station {target_station[:5]}...')
ax1.set_title(f'Full Time Series: AOD (Hourly Mean) - Station {target_station[:5]}...', fontsize=14)
ax1.set_ylabel('Aerosol Optical Depth (AOD)')
ax1.set_xlabel('Date')
ax1.legend()

# Format X-axis for dates
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator())


# --- Plot 2: Specific Day Zoom (e.g., Jan 2, 2025) ---
specific_day = '2025-08-02'

# Safely extract day_data from df_plot (works for DatetimeIndex or 'timestamp' column)
if isinstance(df_plot.index, pd.DatetimeIndex):
    try:
        # label-based slicing (returns Series for single-column; DataFrame for multiple)
        day_data = df_plot.loc[specific_day]
        if isinstance(day_data, pd.Series):
            day_data = day_data.to_frame().T
    except KeyError:
        # fallback to explicit range selection
        start = pd.to_datetime(specific_day)
        end = start + pd.Timedelta(days=1)
        day_data = df_plot[(df_plot.index >= start) & (df_plot.index < end)].copy()
else:
    # fallback: look for a timestamp column
    if 'timestamp' in df_plot.columns:
        df_plot['timestamp'] = pd.to_datetime(df_plot['timestamp'])
        start_date = pd.to_datetime(specific_day).date()
        day_data = df_plot[df_plot['timestamp'].dt.date == start_date].copy()
        if not day_data.empty:
            day_data.set_index('timestamp', inplace=True)
    else:
        day_data = pd.DataFrame()

# Check if data exists for this day
if not day_data.empty:
    # Plot markers to show individual data points, line connects them (breaking at NaNs)
    ax2.plot(day_data.index, day_data['AOD'], color='tab:red', marker='o', markersize=4, linestyle='-', linewidth=2)
    
    ax2.set_title(f'Specific Day View: {specific_day} (10-min Intervals)', fontsize=14)
    ax2.set_ylabel('AOD Value')
    ax2.set_xlabel('Time of Day')
    
    # Format X-axis to show hours
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax2.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    ax2.grid(True, which='both', linestyle='--', alpha=0.7)
    
    # Highlight the concept of gaps
    ax2.text(0.02, 0.95, 'Note: Broken lines indicate missing data (NaN)', transform=ax2.transAxes, 
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))
else:
    ax2.text(0.5, 0.5, f"No data available for {specific_day}", ha='center', fontsize=12)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

# Make a safe copy and ensure numeric types for plotting/regression
df_plot = df_merged.copy()
df_plot['AQI'] = pd.to_numeric(df_plot['AQI'], errors='coerce')
df_plot['AOD'] = pd.to_numeric(df_plot['AOD'], errors='coerce')

# Drop rows with missing or non-numeric values
df_plot = df_plot.dropna(subset=['AQI', 'AOD'])

# Scatter plot with regression line
sns.regplot(
    data=df_plot, 
    x='AOD', 
    y='AQI', 
    scatter_kws={'alpha':0.3, 's':10}, # Light transparency to see density
    line_kws={'color':'red'}
)

# Calculate Correlation Coefficient
corr = df_plot['AQI'].corr(df_plot['AOD'])

plt.title(f'Overall Relationship: AQI vs AOD (Pearson Corr: {corr:.2f})', fontsize=15)
plt.xlabel('Aerosol Optical Depth (AOD) - Hourly Mean')
plt.ylabel('Air Quality Index (AQI)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Aggregate to Daily Average for cleaner time series visualization
# Ensure numeric types for aggregation and use datetime index + resample to avoid .dt.date grouping issues
df_tmp = df_merged.copy()
df_tmp['AQI'] = pd.to_numeric(df_tmp['AQI'], errors='coerce')
df_tmp['AOD'] = pd.to_numeric(df_tmp['AOD'], errors='coerce')

# set Timestamp_DT as index (already datetime) and resample daily
df_tmp = df_tmp.set_index('Timestamp_DT')
df_daily = df_tmp.resample('D')[['AQI', 'AOD']].mean()

fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot AQI on Left Axis
color = 'tab:blue'
ax1.set_xlabel('Date')
ax1.set_ylabel('Avg AQI', color=color, fontsize=12)
ax1.plot(df_daily.index, df_daily['AQI'], color=color, linewidth=2, label='AQI')
ax1.tick_params(axis='y', labelcolor=color)

# Create Right Axis for AOD
ax2 = ax1.twinx()  
color = 'tab:orange'
ax2.set_ylabel('Avg AOD', color=color, fontsize=12)
ax2.plot(df_daily.index, df_daily['AOD'], color=color, linewidth=2, linestyle='--', label='AOD')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Temporal Trend: Daily Average AQI vs AOD (12 Months)', fontsize=15)
fig.tight_layout()  
plt.show()

In [ ]:
# Create a FacetGrid to draw a plot for each Station ID
g = sns.lmplot(
    data=df_merged, 
    x="AOD", 
    y="AQI", 
    col="ID",        # Separate graphs by Station ID
    col_wrap=4,      # Wrap after 4 graphs
    height=3.5, 
    aspect=1,
    scatter_kws={'alpha': 0.3, 's': 10},
    line_kws={'color': 'red'}
)

g.fig.suptitle('AQI vs AOD Relationship by Station', y=1.02, fontsize=16)

plt.show()

In [ ]:
plt.figure(figsize=(10, 8))

# Hexbin plot
hb = plt.hexbin(
    df_merged['AOD'], 
    df_merged['AQI'], 
    gridsize=30,     # Size of the hexagons
    cmap='inferno',  # Color map
    mincnt=1         # Ignore empty hexagons
)

cb = plt.colorbar(hb, label='Count of Records')
plt.title('Density Plot: Concentration of AQI/AOD Values', fontsize=15)
plt.xlabel('Aerosol Optical Depth (AOD)')
plt.ylabel('Air Quality Index (AQI)')
plt.show()

In [ ]:
# E. Clean Missing Values
# Drop rows where we don't have a valid pair of AQI and AOD
df_final = df_merged.dropna(subset=['AQI', 'AOD'])

In [ ]:
df_final